# 👉 Урок 13 & 14. Улучшение векторного поиска и оценка RAG-систем

## 🎯 Цели урока

- понять, почему стандартный поиск (KNN) перестаёт работать с большими данными;
- изучить библиотеку FAISS для масштабируемого векторного поиска;
- освоить ключевые методы ускорения: индексация, квантование, кластеризация;
- научиться оценивать качество RAG-систем с помощью математических, бизнес-метрик и LLM-as-a-Judge.

---

Автор курса: Логинов Дмитрий Владимирович, преподаватель и методист МШП.


# 📈 Часть 1. Проблема масштабирования поиска

В прошлых уроках мы обсуждали, как создаются эмбеддинги и как мы можем искать похожие документы.

Однако есть важная проблема.

## Стандартный подход: KNN

При использовании K-Nearest Neighbors (KNN) мы сравниваем вектор запроса со всеми векторами в базе данных.

Это называется **полным перебором (Brute-Force)**.

### Почему это плохо?

```text
Время поиска = O(n * d)
```

где:
- n — количество документов;
- d — размерность эмбеддинга.

Если у нас:

```text
n = 1,000,000 документов
d = 768 (размерность эмбеддинга)
```

То каждый запрос требует **768 миллионов операций**.

Это слишком медленно для реальных приложений.

## Выход есть

Вместо точного поиска мы используем **приближённый поиск ближайших соседей (ANN — Approximate Nearest Neighbors)**.

Он позволяет:

- ускорить поиск в сотни раз;
- сохранить высокую точность;
- работать с миллиардами документов.

**FAISS (Facebook AI Similarity Search)** — одна из лучших библиотек для ANN.

# 🧩 Часть 2. Основы FAISS

## Определение

FAISS поддерживает:

- индексацию на GPU и CPU;
- различные типы индексов для разных задач;
- квантование для сжатия памяти;
- кластеризацию для ускорения поиска.

## Установка

```bash
pip install faiss-cpu   # Для CPU
# или
pip install faiss-gpu   # Для GPU (требуется CUDA)
```

## Три ключевых понятия

### 1. Индекс

Структура данных, которая хранит векторы и позволяет быстро искать по ним.

### 2. Квантование

Метод сжатия векторов для уменьшения использования памяти.

### 3. Кластеризация

Разбиение данных на группы (кластеры) для ускорения поиска.


# ⚙️ Часть 2.1. Принцип работы FAISS

## Базовая архитектура

FAISS работает по следующему принципу:

```text
1. Подготовка данных
   └── Векторы эмбеддингов загружаются в память

2. Построение индекса
   ├── Выбор типа индекса (Flat, IVF, PQ, HNSW и т.д.)
   ├── Обучение индекса (для IVF/PQ — кластеризация и квантование)
   └── Добавление векторов в индекс

3. Поиск
   ├── Вектор запроса преобразуется в тот же формат
   ├── Вычисление расстояний до векторов в индексе
   │   ├── Евклидово расстояние (L2)
   │   └── Косинусное сходство (IP — Inner Product)
   └── Возврат топ-K ближайших векторов
```

## Важные аспекты работы

### Предобработка векторов

```text
Исходные векторы → Нормализация (опционально) → Добавление в индекс
```

### Поиск с помощью ANN

Приближённый поиск работает быстрее за счёт:

1. **Кластеризации** — поиск только в ближайших кластерах;
2. **Квантования** — сжатие векторов для быстрого сравнения;
3. **Графовых структур** — использование HNSW для быстрого перебора.

### Параллелизация

FAISS поддерживает:
- многопоточность на CPU;
- вычисления на GPU (через CUDA);
- батчевую обработку запросов.

## Пример жизненного цикла индекса

```python
# 1. Создание индекса
index = faiss.IndexIVFFlat(quantizer, d, nlist)

# 2. Обучение (только для некоторых типов)
index.train(training_data)

# 3. Добавление данных
index.add(data)

# 4. Настройка параметров поиска
index.nprobe = 10  # количество кластеров для поиска

# 5. Поиск
distances, indices = index.search(query, k)
```

## Особенности хранения

FAISS хранит векторы в:
- оперативной памяти (быстро);
- на диске (для больших данных, но медленнее);
- в сжатом виде (квантование).


In [ ]:
!pip install faiss-cpu

In [ ]:
# 🔧 Пример 1: Создание простого индекса в FAISS

import numpy as np
import faiss

# 1. Создаём случайные данные
d = 128                  # размерность векторов
n = 10000                # количество векторов
np.random.seed(42)
data = np.random.random((n, d)).astype('float32')

# 2. Создаём индекс Flat L2 (точный поиск по евклидову расстоянию)
index = faiss.IndexFlatL2(d)

# 3. Добавляем данные в индекс
index.add(data)

print(f"Количество векторов в индексе: {index.ntotal}")

# 4. Поиск
query = np.random.random((1, d)).astype('float32')
k = 5  # ищем 5 ближайших соседей
distances, indices = index.search(query, k)

print(f"Индексы ближайших векторов: {indices[0]}")
print(f"Расстояния до них: {distances[0]}")

Количество векторов в индексе: 10000
Индексы ближайших векторов: [8769 9385   82 5125 9571]
Расстояния до них: [13.346977 14.548462 14.70829  14.756073 14.837166]


# 🚀 Часть 3. Типы индексов FAISS

## IndexFlat

**Точный, но медленный.**

```text
IndexFlatL2  → Евклидово расстояние
IndexFlatIP  → Косинусное сходство (Inner Product)
```

Используйте только для маленьких баз данных (< 100k векторов).

## IndexIVF (Inverted File)

**Кластеризация + поиск в ближайших кластерах.**

Схема работы:

```text
Все векторы
    ↓
Кластеризация (nlist кластеров)
    ↓
Поиск: проверяем только nprobe ближайших кластеров
```

Это даёт ускорение в 10-100 раз.

## IndexPQ (Product Quantization)

**Сжатие векторов.**

Вектор разбивается на подвекторы, каждый квантуется отдельно.

Память уменьшается в 10-20 раз.

## IndexHNSW

**Иерархический граф.**

Очень быстрый поиск, но медленное добавление новых векторов.

## Комбинации индексов

Часто используют комбинации:

```text
IVF + PQ → быстрый и компактный
HNSW + Flat → очень быстрый, но требует много памяти
IVF + Flat → быстрее, чем Flat, но точнее, чем PQ
```

In [ ]:
# 🔧 Пример 2: Сравнение скорости и точности разных индексов

import numpy as np
import faiss
import time

d = 128
n = 1000000
np.random.seed(42)
data = np.random.random((n, d)).astype('float32')
query = np.random.random((10, d)).astype('float32')
k = 10

# 1. Точный поиск (Flat)
index_flat = faiss.IndexFlatL2(d)
index_flat.add(data)

start = time.time()
dist_flat, idx_flat = index_flat.search(query, k)
time_flat = time.time() - start

# 2. IVF (кластеризация)
nlist = 100  # количество кластеров
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
index_ivf.train(data)      # обучение кластеров
index_ivf.add(data)
index_ivf.nprobe = 10      # проверяем только 10 ближайших кластеров

start = time.time()
dist_ivf, idx_ivf = index_ivf.search(query, k)
time_ivf = time.time() - start

# 3. IVF + PQ (кластеризация + квантование)
m = 16  # количество подвекторов
quantizer = faiss.IndexFlatL2(d)
index_pq = faiss.IndexIVFPQ(quantizer, d, nlist, m, 8)
index_pq.train(data)
index_pq.add(data)
index_pq.nprobe = 10

start = time.time()
dist_pq, idx_pq = index_pq.search(query, k)
time_pq = time.time() - start

# Результаты
print(f"Flat:    {time_flat:.4f} сек")
print(f"IVF:     {time_ivf:.4f} сек")
print(f"IVF+PQ:  {time_pq:.4f} сек")

Flat:    1.2770 сек
IVF:     0.0414 сек
IVF+PQ:  0.0240 сек


# 📊 Часть 4. Метрики оценки RAG-систем

Теперь, когда мы умеем строить быстрый поиск, возникает новый вопрос:

> Как понять, что наша RAG-система работает хорошо?

Для этого используются метрики.

## Классификация метрик

```text
Метрики RAG
    ├── Математические (автоматические)
    │   ├── Точность поиска (Recall, MRR, NDCG)
    │   └── Качество генерации (ROUGE, BLEU, BERTScore)
    ├── Бизнес-метрики
    │   ├── Время ответа (Latency)
    │   ├── Коэффициент использования (Adoption Rate)
    │   └── Точность ответа пользователя (User Accuracy)
    └── LLM as a Judge
        ├── GPT-4 оценивает ответы
        └── Сравнение с эталоном
```


# 📐 Часть 5. Математические метрики

## Метрики поиска (Retrieval Metrics)

### Recall@k

Какая доля релевантных документов попала в топ-k?

```text
Recall@k = (Релевантные документы среди топ-k) / (Все релевантные документы)
```

**Пример:**

```text
Всего релевантных документов: 10
Среди топ-5 найдено: 7
Recall@5 = 7/10 = 0.7
```

### MRR (Mean Reciprocal Rank)

Где находится первый релевантный документ?

```text
MRR = average(1 / rank первого релевантного документа)
```

**Пример:**

```text
Запрос 1: первый релевантный на позиции 2 → 1/2 = 0.5
Запрос 2: первый релевантный на позиции 1 → 1/1 = 1.0
Запрос 3: первый релевантный на позиции 4 → 1/4 = 0.25
MRR = (0.5 + 1.0 + 0.25) / 3 = 0.58
```

### NDCG (Normalized Discounted Cumulative Gain)

Учитывает и позицию, и релевантность (ранжирование).

```text
DCG = Σ (relevance_i / log2(i + 1))
NDCG = DCG / IDCG (идеальный DCG)
```

## Метрики генерации (Generation Metrics)

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

Сравнивает ответ с эталоном по совпадению n-грамм.

```text
ROUGE-N = (совпавшие n-граммы) / (n-граммы в эталоне)
```

### BLEU

Используется в машинном переводе, но применим и к RAG.

```text
BLEU = точность совпадения n-грамм с штрафом за длину
```

### BERTScore

Использует BERT для семантического сравнения.

```text
Более устойчив к перефразировке, чем ROUGE и BLEU.
```

In [ ]:
# 🔧 Пример 3: Расчёт метрик поиска

import numpy as np

# Данные по одному запросу
relevant = {1, 3, 5, 7, 9}  # индексы релевантных документов
retrieved = [1, 2, 3, 4, 5]  # топ-5 результатов поиска

# Recall@5
retrieved_set = set(retrieved)
recall = len(retrieved_set & relevant) / len(relevant)
print(f"Recall@5: {recall:.2f}")

# Precision@5
precision = len(retrieved_set & relevant) / len(retrieved)
print(f"Precision@5: {precision:.2f}")

# MRR
def mrr(relevant, retrieved):
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant:
            return 1.0 / i
    return 0.0

rr = mrr(relevant, retrieved)
print(f"Reciprocal Rank: {rr:.2f}")

# NDCG (пример)
def dcg(relevances):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))

# Релевантность для топ-5
relevances = [1, 0, 1, 0, 1]  # документы 1, 3, 5 релевантны
ideal_relevances = sorted(relevances, reverse=True)

dcg_value = dcg(relevances)
idcg_value = dcg(ideal_relevances)
ndcg = dcg_value / idcg_value
print(f"NDCG@5: {ndcg:.2f}")

Recall@5: 0.60
Precision@5: 0.60
Reciprocal Rank: 1.00
NDCG@5: 0.89


# 💼 Часть 6. Бизнес-метрики

Математические метрики — это хорошо, но бизнесу важно другое.

## Основные бизнес-метрики

### 1. Latency (Задержка)

Сколько времени проходит от запроса до ответа.

```text
Хорошо:   < 1 сек
Приемлемо: 1-3 сек
Плохо:    > 5 сек
```

### 2. Throughput (Пропускная способность)

Сколько запросов система может обработать за минуту.

### 3. Adoption Rate (Коэффициент использования)

Сколько пользователей реально используют систему.

```text
Adoption Rate = (Активные пользователи) / (Все пользователи)
```

### 4. User Accuracy (Точность ответа пользователя)

Пользователь сам оценивает, помог ли ему ответ.

Способы сбора:
- лайки/дизлайки;
- оценка по шкале;
- отслеживание повторных обращений.

### 5. Cost per Query (Стоимость запроса)

Сколько денег уходит на один запрос.

```text
Cost = (embedding_cost + search_cost + generation_cost) / количество запросов
```


# 📋 Часть 6.1. Детальная классификация метрик по категориям

## 1. Метрики качества поиска (Retrieval Quality)

### 1.1. Точность (Precision)
- **Precision@k**: доля релевантных документов среди топ-k
- **Average Precision (AP)**: средняя точность на разных уровнях recall
- **Mean Average Precision (MAP)**: среднее AP по всем запросам

### 1.2. Полнота (Recall)
- **Recall@k**: доля всех релевантных документов, попавших в топ-k
- **Recall@R**: полнота при фиксированном количестве документов R

### 1.3. Ранжирование (Ranking)
- **MRR**: средняя обратная позиция первого релевантного документа
- **NDCG**: нормализованный дисконтированный кумулятивный выигрыш
- **MAP**: средняя точность с учётом ранжирования

### 1.4. Комбинированные
- **F1@k**: гармоническое среднее Precision и Recall при k
- **R-Precision**: точность при R = количество релевантных документов

## 2. Метрики качества генерации (Generation Quality)

### 2.1. N-грамные метрики
- **ROUGE-N**: совпадение n-грамм (N=1,2,3,4)
- **ROUGE-L**: совпадение по longest common subsequence
- **BLEU**: точность n-грамм с штрафом за длину
- **METEOR**: учитывает синонимы и стемминг

### 2.2. Семантические метрики
- **BERTScore**: семантическая схожесть через BERT
- **BLEURT**: обученная метрика на основе BERT
- **COMET**: метрика для оценки качества перевода

### 2.3. Метрики разнообразия
- **Distinct-N**: доля уникальных n-грамм в ответе
- **Entropy**: энтропия распределения слов

## 3. Метрики эффективности (Performance Metrics)

### 3.1. Временные метрики
- **P99 Latency**: 99-й перцентиль времени ответа
- **Average Latency**: среднее время ответа
- **Time to First Token (TTFT)**: время до первого токена

### 3.2. Пропускная способность
- **Requests Per Second (RPS)**: запросов в секунду
- **Tokens Per Second (TPS)**: токенов в секунду

### 3.3. Ресурсные метрики
- **Memory Usage**: использование памяти
- **GPU Utilization**: загрузка GPU
- **CPU Utilization**: загрузка CPU

## 4. Бизнес-метрики (Business Metrics)

### 4.1. Пользовательские метрики
- **User Retention**: удержание пользователей
- **Adoption Rate**: доля активных пользователей
- **Churn Rate**: отток пользователей
- **User Satisfaction Score (USS)**: удовлетворённость пользователей

### 4.2. Финансовые метрики
- **Cost per Query**: стоимость одного запроса
- **Cost per User**: стоимость одного пользователя
- **ROI**: возврат инвестиций
- **Revenue per User**: доход на пользователя

### 4.3. Операционные метрики
- **Error Rate**: доля ошибок в работе системы
- **Availability**: доступность системы (Uptime)
- **SLA Compliance**: соблюдение уровней обслуживания

## 5. Метрики надёжности (Reliability Metrics)

- **Hallucination Rate**: доля галлюцинаций в ответах
- **Factual Consistency**: фактическая согласованность с источниками
- **Toxicity Score**: уровень токсичности ответов
- **Bias Score**: уровень предвзятости

## 6. Метрики для LLM as a Judge

- **Coherence**: связность ответа
- **Relevance**: релевантность запросу
- **Completeness**: полнота ответа
- **Helpfulness**: полезность для пользователя
- **Safety**: безопасность контента
- **Instruction Following**: следование инструкциям
- **Clarity**: ясность и понятность


# 🤖 Часть 7. LLM as a Judge

Это современный и очень мощный подход.

## Определение

**LLM as a Judge** — техника, при которой сильная модель (например, GPT-4) оценивает ответы другой модели.

## Почему это работает?

LLM хорошо понимают:

- семантику;
- логику;
- стиль;
- полезность.

Они могут оценивать ответы по разным критериям.

## Основные критерии оценки

| Критерий | Описание |
|----------|----------|
| Coherence | Связность и логичность |
| Relevance | Соответствие запросу |
| Factual Accuracy | Фактическая точность |
| Completeness | Полнота ответа |
| Helpfulness | Полезность для пользователя |
| Safety | Безопасность контента |

## Преимущества

- не требуется размеченных данных;
- быстрая оценка;
- гибкость (можно менять критерии);
- масштабируемость.

## Недостатки

- дорого (используется GPT-4);
- медленно;
- возможна предвзятость LLM;
- сложно воспроизвести.

## Пример использования

```text
Системный промпт для Judge:
Ты — эксперт по оценке качества ответов.
Оцени ответ по шкале от 1 до 5 по следующим критериям:
- точность,
- полнота,
- полезность.
Дай краткое обоснование.

Запрос: [вопрос пользователя]
Ответ: [ответ модели]

Оценка:
```


In [ ]:
# 🔧 Пример 4: LLM as a Judge с OpenRouter

import openai

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="",
)

def judge_response(query, response):
    """Оценивает ответ с помощью LLM"""
    prompt = f"""
    Ты — эксперт по оценке качества ответов RAG-систем.
    Оцени ответ по шкале от 1 до 5 по трём критериям:
    1. Точность (фактическая правильность)
    2. Полнота (все ли аспекты вопроса раскрыты)
    3. Полезность (насколько ответ помогает пользователю)

    Вопрос пользователя: {query}
    Ответ модели: {response}

    Верни ответ в формате:
    Точность: X/5
    Полнота: X/5
    Полезность: X/5
    Итоговая оценка: X/5
    Обоснование: [краткий комментарий]
    """

    judge_response = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )
    return judge_response.choices[0].message.content

# Пример использования
query = "Что такое RAG?"
response = "RAG — это Retrieval-Augmented Generation, метод, который сочетает поиск документов и генерацию текста."

score = judge_response(query, response)
print(score)

Точность: 5/5
Полнота: 3/5
Полезность: 3/5
Итоговая оценка: 3.7/5
Обоснование: Ответ фактически верен и дает корректное расшифровку аббревиатуры и суть метода. Однако он неполный: отсутствует объяснение *зачем* это нужно (решение проблем галлюцинаций, актуальность знаний), *как* именно работает пайплайн (ретривер -> генератор) и примеры применения. Для пользователя, не знакомого с темой, этого определения недостаточно для понимания ценности технологии.


# 📊 Часть 8. Сравнение метрик

## Когда какую метрику использовать?

| Ситуация | Метрика |
|----------|---------|
| Оценка качества поиска | Recall@k, MRR, NDCG |
| Сравнение с эталонным ответом | ROUGE, BLEU, BERTScore |
| Пользовательский опыт | User Accuracy, Adoption Rate |
| Бизнес-эффективность | Latency, Cost per Query |
| Быстрая оценка без эталона | LLM as a Judge |

## Рекомендации

1. **Всегда используйте несколько метрик.**
   - Одна метрика не даёт полной картины.

2. **Используйте автоматические метрики на этапе разработки.**
   - Они быстрые и дешёвые.

3. **Используйте LLM as a Judge для финальной оценки.**
   - Она ближе к человеческой оценке.

4. **Отслеживайте бизнес-метрики в продакшене.**
   - Именно они показывают реальную ценность системы.


# 🎯 Заключение

## Что мы узнали?

1. **Проблема масштабирования поиска**
   - Brute-Force становится слишком медленным.

2. **FAISS**
   - Библиотека для быстрого ANN-поиска.
   - Поддерживает различные индексы: Flat, IVF, PQ, HNSW.
   - Основа работы: кластеризация, квантование и графовые структуры.

3. **Метрики RAG**
   - Математические: Recall@k, MRR, NDCG, ROUGE, BLEU, BERTScore.
   - Бизнес-метрики: Latency, Throughput, Adoption Rate, Cost.
   - LLM as a Judge: гибкая и мощная техника оценки.
   - Детальная классификация по 6 категориям: качество поиска, качество генерации, эффективность, бизнес, надёжность и LLM-оценка.

## Следующие шаги

- Оптимизация индексов под вашу задачу;
- A/B-тестирование разных моделей;
- Сбор пользовательской обратной связи;
- Построение дашбордов для мониторинга.
